# ER1: No-Comm Control Baseline

**Objective:** Establish the performance floor by training agents with **zero communication** on the Discovery K-N convergence task. Any communication method must beat this to justify its existence.

**What this isolates:** Pure task difficulty under partial observability — agents can only see via LiDAR, no message passing.

**Sweep:** N ∈ {2,3,4,6} agents × K=2 × lidar_range ∈ {0.25, 0.35, 0.45} × {MAPPO, IPPO} × 5 seeds

**Pass/Fail:** Floor established at 30–40% success (N=4, K=2). If E1+ methods don't beat this → communication is unnecessary.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add rendezvous_comm src to path
REPO_ROOT = Path.cwd().parent.parent
RENDEZVOUS_ROOT = REPO_ROOT / "rendezvous_comm"
sys.path.insert(0, str(RENDEZVOUS_ROOT))

# Core imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Experiment infrastructure
from src.config import load_experiment, ExperimentSpec, TaskConfig, TrainConfig
from src.metrics import EpisodeMetrics
from src.storage import ExperimentStorage, RunStorage
from src.runner import (
    build_experiment, run_single, run_sweep,
    evaluate_with_vmas, make_heuristic_policy_fn,
)
from src.plotting import (
    plot_sweep_heatmap, plot_seed_variance,
    plot_baseline_comparison, save_figure,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load Experiment Config

In [ ]:
# Load from YAML config
spec = load_experiment(RENDEZVOUS_ROOT / "configs" / "er1_no_comm.yaml")
spec.ensure_dirs()

print(f"Experiment: {spec.exp_id} — {spec.name}")
print(f"Description: {spec.description}")
print(f"\nTask config: {spec.task.to_dict()}")
print(f"\nSweep: {spec.sweep}")
print(f"\nTotal runs: {sum(1 for _ in spec.iter_runs())}")

## 3. Quick Sanity Check — Heuristic Baseline

Before training, verify the environment works and establish a heuristic reference point. The discovery scenario has a built-in `HeuristicPolicy` (circular patrol + greedy target-seeking).

In [ ]:
# Quick heuristic eval (no training, ~10 seconds)
heuristic_fn = make_heuristic_policy_fn()
heuristic_metrics = evaluate_with_vmas(
    spec.task,
    task_overrides={"n_agents": 4, "n_targets": 7, "agents_per_target": 2},
    policy_fn=heuristic_fn,
    n_eval_episodes=200,
    n_envs=200,
)
print("Heuristic baseline (no training):")
for k, v in heuristic_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Also check random baseline for comparison
random_metrics = evaluate_with_vmas(
    spec.task,
    task_overrides={"n_agents": 4, "n_targets": 7, "agents_per_target": 2},
    policy_fn=None,  # random policy
    n_eval_episodes=200,
    n_envs=200,
)
print("Random baseline:")
for k, v in random_metrics.items():
    print(f"  {k}: {v:.4f}")

## 4. Training — BenchMARL Sweep

Run the full parameter sweep. Each run trains MAPPO/IPPO for 10M frames.

**Estimated time:** ~2-4 hours per run on CPU, ~30 min on GPU. With 120 total runs (4 agents × 3 lidar × 2 algos × 5 seeds), consider:
- Running on a cluster / multiple GPUs
- Using `skip_complete=True` to resume interrupted sweeps
- Starting with a smaller `max_n_frames` for quick iteration

In [ ]:
# ── Option A: Quick test run (small frames, 1 config) ──────────────
# Uncomment to do a quick sanity check before full sweep

# spec.train.max_n_frames = 60_000  # ~1 iteration
# spec.train.on_policy_n_envs_per_worker = 60
# spec.train.on_policy_n_minibatch_iters = 1

# test_metrics = run_single(
#     spec,
#     run_id="er1_test_mappo_n_agents4_s0",
#     task_overrides={"n_agents": 4},
#     algorithm="mappo",
#     seed=0,
#     dry_run=False,
# )
# print(test_metrics)

In [ ]:
# ── Option B: Full sweep ───────────────────────────────────────────
# This runs ALL parameter combinations. Use skip_complete=True to resume.

# results = run_sweep(spec, skip_complete=True, dry_run=False)
# print(f"\nCompleted {len(results)} runs")

In [ ]:
# ── Option C: Run a single specific configuration ─────────────────
# Useful for targeted runs or debugging

from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig
from benchmarl.algorithms import MappoConfig, IppoConfig
from benchmarl.models.mlp import MlpConfig

# Choose configuration
ALGORITHM = "mappo"  # "mappo" or "ippo"
N_AGENTS = 4
SEED = 0
DEVICE = "cpu"  # "cpu" or "cuda"

# Task
task = VmasTask.DISCOVERY.get_from_yaml()
task.config.update({
    "n_agents": N_AGENTS,
    "n_targets": 7,
    "agents_per_target": 2,
    "lidar_range": 0.35,
    "covering_range": 0.25,
    "use_agent_lidar": False,       # No agent-agent comm/sensing
    "targets_respawn": True,
    "shared_reward": False,
    "agent_collision_penalty": -0.1,
    "time_penalty": -0.01,
})

# Algorithm
algo_config = MappoConfig.get_from_yaml() if ALGORITHM == "mappo" else IppoConfig.get_from_yaml()

# Model
model_config = MlpConfig.get_from_yaml()
critic_model_config = MlpConfig.get_from_yaml()

# Experiment config
experiment_config = ExperimentConfig.get_from_yaml()
experiment_config.max_n_frames = 10_000_000
experiment_config.train_device = DEVICE
experiment_config.sampling_device = DEVICE
experiment_config.gamma = 0.99
experiment_config.on_policy_collected_frames_per_batch = 60_000
experiment_config.on_policy_n_envs_per_worker = 600
experiment_config.on_policy_n_minibatch_iters = 45
experiment_config.on_policy_minibatch_size = 4096
experiment_config.share_policy_params = True
experiment_config.evaluation = True
experiment_config.evaluation_interval = 120_000
experiment_config.evaluation_episodes = 200
experiment_config.loggers = ["csv"]

print(f"Config: {ALGORITHM}, N={N_AGENTS}, seed={SEED}, device={DEVICE}")
print(f"Total training frames: {experiment_config.max_n_frames:,}")

In [ ]:
# Train!
experiment = Experiment(
    task=task,
    algorithm_config=algo_config,
    model_config=model_config,
    critic_model_config=critic_model_config,
    seed=SEED,
    config=experiment_config,
)
experiment.run()

## 5. Post-Training Evaluation

Load results and compute the final metrics (M1-M4) across all completed runs.

In [ ]:
# Load all completed run metrics
storage = ExperimentStorage("er1")
df = storage.to_dataframe()

if df.empty:
    print("No completed runs yet. Train first (Section 4).")
else:
    print(f"Loaded {len(df)} completed runs")
    display(df.describe())

## 6. Analysis & Plots

In [ ]:
# ── 6a. Success rate heatmap: n_agents × lidar_range ──────────────
if not df.empty and "n_agents" in df.columns:
    # Parse n_agents and lidar_range from run_id if not already columns
    fig = plot_sweep_heatmap(
        df, metric="M1_success_rate",
        row_param="n_agents", col_param="lidar_range",
        title="ER1 No-Comm: Success Rate by Agents × LiDAR Range",
    )
    save_figure(fig, str(spec.results_dir / "er1_success_heatmap.png"))
    plt.show()

In [ ]:
# ── 6b. MAPPO vs IPPO comparison (seed variance) ─────────────────
if not df.empty and "algorithm" in df.columns:
    fig = plot_seed_variance(
        df, metric="M1_success_rate", group_by="algorithm",
        title="ER1: MAPPO vs IPPO Success Rate (across seeds)",
    )
    save_figure(fig, str(spec.results_dir / "er1_algo_comparison.png"))
    plt.show()

In [ ]:
# ── 6c. All four metrics as a summary table ──────────────────────
if not df.empty:
    summary = df.groupby(["algorithm"]).agg({
        "M1_success_rate": ["mean", "std"],
        "M2_avg_return": ["mean", "std"],
        "M3_avg_steps": ["mean", "std"],
        "M4_avg_collisions": ["mean", "std"],
    }).round(4)
    display(summary)

## 7. Pass/Fail Assessment

**Expected:** Floor at 30–40% success for N=4, K=2.  
**Implication:** If subsequent experiments (ER2–E1) do NOT beat this floor, communication adds no value for this task configuration.

In [ ]:
# ── Pass/Fail check ───────────────────────────────────────────────
if not df.empty:
    n4_mask = df["run_id"].str.contains("n_agents4") if "n_agents" not in df.columns else df["n_agents"] == 4
    n4_success = df.loc[n4_mask, "M1_success_rate"].mean()

    print(f"N=4, K=2 average success rate: {n4_success:.2%}")
    print()
    if 0.20 <= n4_success <= 0.50:
        print("PASS: Floor established in expected range (20-50%).")
        print("Proceed to ER2+ experiments — comm methods must beat this.")
    elif n4_success > 0.50:
        print("NOTE: Floor higher than expected. Consider:")
        print("  - Increasing n_targets or agents_per_target to make task harder")
        print("  - Reducing lidar_range to increase partial observability")
    else:
        print("NOTE: Floor lower than expected. Consider:")
        print("  - Training for more frames")
        print("  - Checking if task is too hard for any method")